# import libraries

In [1]:
import pandas as pd
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.factory import Models
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy   
import time


import sys
sys.path.append('../')
import helper_functions as hf

# generate recommendations

In [2]:
n = hf.get_iteration_number()

for i in range(3):
    print("Very Important: Please Confirm the Iteration Number is Iteration " + str(n))

Very Important: Please Confirm the Iteration Number is Iteration 1
Very Important: Please Confirm the Iteration Number is Iteration 1
Very Important: Please Confirm the Iteration Number is Iteration 1


In [3]:
time_start = time.time()


df_design, ax_client = hf.run_optimizer(current_iteration=n, drug = "IBP", bopt = 1,n_trials=6)

time_end = time.time()
time_duration = round((time_end - time_start)/60,2)

print("Time taken for optimization: " + str(time_duration) + " mins")
print("Time taken for optimization: " + str(time_duration * 60) + " seconds")

[INFO 06-20 10:44:15] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-20 10:44:16] ax.modelbridge.transforms.standardize_y: Outcome complexity is constant, within tolerance.


**************************************************************************************************************

Generating Bayesian Optimization trialsfor
Drug name:  Ibuprofen IBP  | Iteration:  1

**************************************************************************************************************


/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/botorch/models/utils/assorted.py:267: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([0.], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)
[INFO 06-20 10:45:03] ax.service.ax_client: Generated new trial 6 with parameters {'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 0, 's2': 0, 's3': 0, 's4': 100, 's5': 0, 's6': 100, 's7': 38, 's8': 0, 'surfactant_conc': 1, 'drug_conc': 100} using model SAASBO.
/opt/anaconda3/envs/drug_surfactant/lib/python3.11/site-packages/ax/core/data.py:295: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relev

Time taken for optimization: 2.77 mins
Time taken for optimization: 166.2 seconds


# process results

In [4]:
ax_client = hf.load_design_optimizer(n)
ax_client.experiment.trials

[INFO 06-20 10:47:27] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.


{0: Trial(experiment_name='drug_surfactant', index=0, status=TrialStatus.COMPLETED, arm=Arm(name='0_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 47, 's2': 59, 's3': 49, 's4': 31, 's5': 96, 's6': 8, 's7': 9, 's8': 35, 'surfactant_conc': 85, 'drug_conc': 77})),
 1: Trial(experiment_name='drug_surfactant', index=1, status=TrialStatus.COMPLETED, arm=Arm(name='1_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 95, 's2': 22, 's3': 76, 's4': 63, 's5': 42, 's6': 71, 's7': 55, 's8': 99, 'surfactant_conc': 35, 'drug_conc': 3})),
 2: Trial(experiment_name='drug_surfactant', index=2, status=TrialStatus.COMPLETED, arm=Arm(name='2_0', parameters={'Drug_MW': 0.206, 'Drug_LogP': 0.307, 'Drug_TPSA': 0.037, 's1': 52, 's2': 98, 's3': 4, 's4': 13, 's5': 22, 's6': 89, 's7': 28, 's8': 56, 'surfactant_conc': 18, 'drug_conc': 33})),
 3: Trial(experiment_name='drug_surfactant', index=3, status=TrialStatus.COMPLETED, arm=Arm(name='3_0', parameter

In [5]:
df_conc, df_vol = hf.design_to_conc_to_vol (n)

In [6]:
plate_well = input("Enter the plate well starting well (e.g., F1): ")
deepplate_well = input("Enter the deep plate well starting well (e.g., F1): ")


print("Please confirm the following information:")
print("Wellplate will start at: " + plate_well)
print("Deep plate will start at: " + deepplate_well)

print()
print("*******************************************************")
print("Continue if correct, or rerun this cell if incorrect.")
print("*******************************************************")

Please confirm the following information:
Wellplate will start at: D1
Deep plate will start at: F1

*******************************************************
Continue if correct, or rerun this cell if incorrect.
*******************************************************


In [7]:

hf.generate_protocol(df_vol=df_vol, iteration=n, plate_well=plate_well, deepplate_well=deepplate_well)

✅ Successfully wrote to: protocol/otflex_1.py


In [8]:
df_absorbance = hf.process_absorbance(iteration=n, threshold=0.1)
df_absorbance

,trial_index,success
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0


In [9]:
results = hf.build_results(n, df_conc, df_absorbance)
results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,6,0,0,0,100,0,100,38,0,0.5,25.00,0,0.0,3
1,7,100,100,100,100,0,0,100,0,50.0,0.25,0,0.0,5
2,8,100,100,100,0,0,0,100,0,0.5,0.25,0,0.0,4
3,9,0,0,100,0,0,100,100,0,0.5,25.00,0,0.0,3
4,10,100,100,0,59,100,100,100,0,0.5,25.00,0,0.0,6
5,11,0,0,0,0,0,0,100,100,0.5,25.00,0,0.0,2


In [10]:
norm_results = hf.normalize_data(results, 'normalize')

In [11]:
norm_results

,trial_index,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc,success,micelle_drug_conc,complexity
0,6,0,0,0,100,0,100,38,0,0.01,25.00,0,0.0,0.375
1,7,100,100,100,100,0,0,100,0,1.00,0.25,0,0.0,0.625
2,8,100,100,100,0,0,0,100,0,0.01,0.25,0,0.0,0.500
3,9,0,0,100,0,0,100,100,0,0.01,25.00,0,0.0,0.375
4,10,100,100,0,59,100,100,100,0,0.01,25.00,0,0.0,0.750
5,11,0,0,0,0,0,0,100,100,0.01,25.00,0,0.0,0.250


# load the results to the optimizer

In [12]:
ax_client = hf.load_data_to_optimizer(iteration = n, norm_results = norm_results)
ax_client

[INFO 06-20 11:30:53] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 06-20 11:30:53] ax.service.ax_client: Completed trial 6 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.375, None)}.
[INFO 06-20 11:30:53] ax.service.ax_client: Completed trial 7 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (1.0, None), 'complexity': (0.625, None)}.
[INFO 06-20 11:30:53] ax.service.ax_client: Completed trial 8 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.5, None)}.
[INFO 06-20 11:30:53] ax.service.ax_client: Completed trial 9 with data: {'micelle_drug_conc': (0.0, None), 'surfactant_conc': (0.01, None), 'complexity': (0.375, None)}.
[INFO 06-20 11:30:53] ax.service.ax_client: Completed trial 10 with data: {'micelle_drug_conc': 

AxClient(experiment=Experiment(drug_surfactant))